# Pseudonymizer — fine-tune on Colab

Before running: **Runtime → Change runtime type → A100 GPU** (Colab Pro). Enable *Background execution* for long runs.

Data (`train.parquet`, `dev.parquet`, `dev.jsonl`, optionally `dev_judges.*`) must be in Google Drive at `MyDrive/ai-camp/silver/`.
Code is cloned from GitHub; every run is copied back to `MyDrive/ai-camp/runs/<name>/`.

Timing on A100, 24k docs × 3 epochs: base models ≈ 30–40 min, large ≈ 1.5 h.

In [ ]:
#@title 1. Code
REPO = "https://github.com/EnderLancer/anonimimizer.git"  #@param {type:"string"}
BRANCH = "dmelashych/datagen"  #@param {type:"string"}
import os
if not os.path.exists('/content/ai-camp'):
    !git clone -q --branch $BRANCH $REPO /content/ai-camp
else:
    !cd /content/ai-camp && git pull -q
%cd /content/ai-camp/train
!pip install -q -e . 2>&1 | tail -1
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch the runtime to GPU')

In [ ]:
#@title 2. Data from Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ai-camp'
!mkdir -p /content/ai-camp/data/silver $DRIVE/runs
!cp $DRIVE/silver/*.parquet $DRIVE/silver/*.jsonl /content/ai-camp/data/silver/
!ls -la /content/ai-camp/data/silver/ && test -f /content/ai-camp/data/evaluate.py && echo 'evaluate.py OK'

In [ ]:
#@title 3. (optional) Hugging Face login — needed only for --push-to
PUSH_TO = ""  #@param {type:"string"}
if PUSH_TO:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get('HF_TOKEN'))  # add HF_TOKEN in Colab: left sidebar → Secrets

In [ ]:
#@title 4. Train
MODEL = "benjamin/roberta-base-wechsel-ukrainian"  #@param ["benjamin/roberta-base-wechsel-ukrainian", "FacebookAI/xlm-roberta-base", "microsoft/mdeberta-v3-base", "benjamin/roberta-large-wechsel-ukrainian", "FacebookAI/xlm-roberta-large"] {allow-input: true}
RUN = "wechsel_base_full"  #@param {type:"string"}
EPOCHS = 3  #@param {type:"number"}
LR = 3e-5  #@param {type:"number"}
BATCH = 32  #@param {type:"integer"}  # A100: 32 (large: 16); T4: 16 (large: 8 + GRAD_ACCUM 2)
GRAD_ACCUM = 1  #@param {type:"integer"}
N_DOCS = 0  #@param {type:"integer"}
DEV_DOCS = 500  #@param {type:"integer"}
extra = (f' --n-docs {N_DOCS}' if N_DOCS else '') + (f' --push-to {PUSH_TO}' if PUSH_TO else '')
!python scripts/train.py --model $MODEL --out runs/$RUN --epochs $EPOCHS --lr $LR --batch $BATCH --grad-accum $GRAD_ACCUM --dev-docs $DEV_DOCS $extra 2>&1 | grep -vE 'it/s\]|s/it\]|HTTP Request|FutureWarning|warnings.warn'

In [ ]:
#@title 4b. (alternative to 4) Train several base models in a row and compare on dev
MODELS = {
    'wechsel_base_full': ('benjamin/roberta-base-wechsel-ukrainian', 3e-5, 32, 1),
    'xlmr_base_full':    ('FacebookAI/xlm-roberta-base',              3e-5, 32, 1),
    'mdeberta_base_full':('microsoft/mdeberta-v3-base',               3e-5, 16, 2),
    # 'wechsel_large_full': ('benjamin/roberta-large-wechsel-ukrainian', 2e-5, 16, 2),
}
import json, subprocess
for run, (model, lr, batch, accum) in MODELS.items():
    print(f'\n===== {run}: {model} =====')
    !python scripts/train.py --model $model --out runs/$run --epochs 3 --lr $lr --batch $batch --grad-accum $accum --dev-docs 500 2>&1 | grep -E 'INFO (dev|train loss)|Error'
    !rm -rf runs/$run/checkpoints && cp -r runs/$run $DRIVE/runs/
print('\n===== leaderboard (dev macro F1 by epoch) =====')
for run in MODELS:
    try:
        h = json.load(open(f'runs/{run}/dev_history.json'))
        print(f"{run:22s}", ' '.join(f"ep{int(x['epoch'])}={x['final_score']:.4f}" for x in h))
    except FileNotFoundError:
        print(run, 'no history')

In [ ]:
#@title 5. Score on full dev with the official script + copy the run to Drive
!python scripts/predict.py --model runs/$RUN/model --input ../data/silver/dev.jsonl --output runs/$RUN/dev_sub.jsonl --batch 32
!python ../data/evaluate.py ../data/silver/dev.jsonl runs/$RUN/dev_sub.jsonl
!python scripts/predict.py --model runs/$RUN/model --input ../data/silver/dev_judges.jsonl --output runs/$RUN/dev_judges_sub.jsonl --batch 32
!python ../data/evaluate.py ../data/silver/dev_judges.jsonl runs/$RUN/dev_judges_sub.jsonl | head -5
!rm -rf runs/$RUN/checkpoints && cp -r runs/$RUN $DRIVE/runs/ && echo copied to $DRIVE/runs/$RUN

In [ ]:
#@title 6. (optional) CPU timing estimate — the organizers' limit is 2 h for 240 docs on CPU
!python scripts/predict.py --model runs/$RUN/model --input ../data/silver/dev.jsonl --output /tmp/cpu_sub.jsonl --device cpu --threads 2 --batch 8 2>&1 | tail -1
print('dev = 500 docs; the test set has 240 docs of similar length')

In [ ]:
#@title 7. FINAL — score on the organizers' test set (run ONCE, for the chosen model only; never tune on this)
CONFIRM = False  #@param {type:"boolean"}
import datetime, subprocess
if CONFIRM:
    !python scripts/predict.py --model runs/$RUN/model --input ../data/test/court_cases.jsonl --output runs/$RUN/test_sub.jsonl --batch 32
    out = subprocess.run(['python', '../data/evaluate.py', '../data/test/court_cases.jsonl', f'runs/{RUN}/test_sub.jsonl'], capture_output=True, text=True).stdout
    print(out)
    with open(f'{DRIVE}/runs/test_runs.log', 'a') as f:
        f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M} {RUN} {out.splitlines()[0]}\n")
    !cp runs/$RUN/test_sub.jsonl $DRIVE/runs/$RUN/
else:
    print('set CONFIRM=True to run; this is the final, logged evaluation')